In [19]:
### imports
import warnings, math
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from torch.utils.data import Dataset

import transformers, accelerate
print("torch        :", torch.__version__)
print("transformers :", transformers.__version__)
print("accelerate   :", accelerate.__version__)
print("CUDA         :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0))
    print("bf16 support :", torch.cuda.is_bf16_supported())

torch        : 2.12.1+cu130
transformers : 5.12.1
accelerate   : 1.14.0
CUDA         : True
GPU          : NVIDIA RTX A4000
bf16 support : True


In [ ]:
## Configuration
SEED        = 42
MODEL_NAME  = "microsoft/deberta-v3-base"
MAX_LEN     = 128
BATCH_SIZE  = 16
NUM_EPOCHS  = 5
LR          = 1e-5          # FIX: lower LR; 2e-5 is too aggressive for DeBERTa cold-start
WARMUP_RATIO = 0.1          # FIX: 10% of steps as warmup to avoid spike on step 1

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = False        # never use fp16 with DeBERTa
print(f"Precision: {'bf16' if USE_BF16 else 'fp32'}")


##ANNOTATED_FILE = Path("/user/HS402/kk01697/Documents/dissertation/story-evaluation-dissertation/data/processed/annotation_sample_annotated.csv")
ANNOTATED_FILE = Path("/scratch/kk01697/data/processed/annotation_sample_annotated.csv")
OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR = OUTPUT_DIR / "deberta_classifier"

CUSTOM_DIM = [
    "Narrative Structure & Quality",
    "Character & Emotion",
    "Originality",
    "Immersion",
    "Thematic Depth",
    "Writing Style",
]

Precision: bf16


In [4]:
ann_df = pd.read_csv(ANNOTATED_FILE)
ann_df = ann_df.dropna(subset=["sentence"])

print("Shape:", ann_df.shape)
ann_df.head()

Shape: (3000, 10)


,review_id,sentence_idx,sentence,language,Narrative Structure & Quality,Character & Emotion,Originality,Immersion,Thematic Depth,Writing Style
0,6c98fe733ae0c27671ebbbe68b77fd8f,6,The initial deepening of the mechanics of the ...,eng,0,0,0,1,0,0
1,cd8abbbf2727515f904a6b189cb0eb84,24,I think that's more troubling when it comes to...,eng,0,0,0,0,0,0
2,b0d8887563f48d59440cd78144ef23c0,59,The one note simple tone of everything leads m...,en-US,0,0,0,0,0,1
3,93060ddc1ef84b111915ed91cfc443de,35,Saving grace was that he was the only characte...,eng,0,1,0,0,0,0
4,4fac8a39591a791eb0a78c4c08176d2b,4,I've never been so disappointed by this author.,eng,0,0,0,0,0,0


In [3]:
class SentenceDataset(Dataset):
    def __init__(self, texts: list[str], labels: np.ndarray, tokenizer):
        self.enc = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.enc["input_ids"][idx],
            "attention_mask": self.enc["attention_mask"][idx],
            "labels":         self.labels[idx],
        }

In [5]:
class WeightedBCETrainer(Trainer):
    """Trainer subclass that injects per-dimension pos_weight into BCE loss."""

    def __init__(self, *args, pos_weight: torch.Tensor = None, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight.to(DEVICE) if pos_weight is not None else None

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits                  # (B, num_labels)

        loss_fn = nn.BCEWithLogitsLoss(pos_weight=self.pos_weight)
        loss    = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [14]:
class NaNGuardTrainer(Trainer):
    """Weighted BCE + NaN gradient guard."""

    def __init__(self, *args, pos_weight: torch.Tensor = None, **kwargs):
        super().__init__(*args, **kwargs)
        self._pos_weight = pos_weight.to(DEVICE) if pos_weight is not None else None
        self._nan_batches = 0

    # ── custom loss ───────────────────────────────────────────────────────────
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits.float()      # force fp32 even in bf16 runs
        labels  = labels.float()

        loss_fn = nn.BCEWithLogitsLoss(pos_weight=self._pos_weight)
        loss    = loss_fn(logits, labels)

        # detect and report NaN loss without crashing
        if torch.isnan(loss):
            self._nan_batches += 1
            print(f"[NaNGuard] NaN loss detected (batch #{self._nan_batches}) — skipping")
            loss = torch.tensor(0.0, requires_grad=True, device=DEVICE)

        return (loss, outputs) if return_outputs else loss

    # ── NaN gradient guard ────────────────────────────────────────────────────
    def training_step(self, model, inputs, num_items_in_batch=None):
        loss = super().training_step(model, inputs, num_items_in_batch)

        # zero any NaN/Inf gradients before the optimizer touches them
        nan_params = []
        for name, param in model.named_parameters():
            if param.grad is not None:
                bad = torch.isnan(param.grad) | torch.isinf(param.grad)
                if bad.any():
                    param.grad[bad] = 0.0
                    nan_params.append(name)
        if nan_params:
            print(f"[NaNGuard] Zeroed NaN/Inf grads in: {nan_params[:3]}{'...' if len(nan_params)>3 else ''}")

        return loss

In [6]:
def build_compute_metrics(threshold: float = 0.5):
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        probs = 1.0 / (1.0 + np.exp(-logits))    # sigmoid
        preds = (probs >= threshold).astype(int)

        micro = f1_score(labels, preds, average="micro", zero_division=0)
        macro = f1_score(labels, preds, average="macro", zero_division=0)
        per   = f1_score(labels, preds, average=None,    zero_division=0)

        metrics = {"f1_micro": micro, "f1_macro": macro}
        for dim, score in zip(CUSTOM_DIM, per):
            metrics[f"f1_{dim}"] = score
        return metrics

    return compute_metrics

In [15]:
torch.manual_seed(SEED)
np.random.seed(SEED)

sentences = ann_df["sentence"].tolist()
labels    = ann_df[CUSTOM_DIM].values.astype(np.float32)

idx = np.arange(len(sentences))
idx_trainval, idx_test = train_test_split(idx, test_size=0.15, random_state=SEED)
idx_train, idx_val     = train_test_split(idx_trainval, test_size=0.15, random_state=SEED)

train_labels = labels[idx_train]
pos_counts   = train_labels.sum(axis=0).clip(min=1)
neg_counts   = len(train_labels) - pos_counts
pos_weight   = torch.tensor(neg_counts / pos_counts, dtype=torch.float32)

print(f"Split — train: {len(idx_train)}  val: {len(idx_val)}  test: {len(idx_test)}\n")
print(f"{'Dimension':<35} {'Pos':>5}  {'pos_weight':>10}")
print("-" * 55)
for dim, p, w in zip(CUSTOM_DIM, pos_counts.astype(int), pos_weight.tolist()):
    print(f"{dim:<35} {p:>5}  {w:>10.1f}")

Split — train: 2167  val: 383  test: 450

Dimension                             Pos  pos_weight
-------------------------------------------------------
Narrative Structure & Quality         335         5.5
Character & Emotion                   417         4.2
Originality                            80        26.1
Immersion                              66        31.8
Thematic Depth                         54        39.1
Writing Style                         125        16.3


In [16]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = SentenceDataset([sentences[i] for i in idx_train], train_labels,      tokenizer)
val_ds   = SentenceDataset([sentences[i] for i in idx_val],   labels[idx_val],   tokenizer)
test_ds  = SentenceDataset([sentences[i] for i in idx_test],  labels[idx_test],  tokenizer)

print("Datasets created.")
print("Sample labels:", train_ds[0]["labels"])

Datasets created.
Sample labels: tensor([0., 0., 0., 0., 0., 0.])


In [17]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(CUSTOM_DIM),
    # problem_type NOT set — NaNGuardTrainer owns the loss
)

# Force all parameters to float32 BEFORE moving to device.
# This prevents the Trainer from silently keeping the backbone in fp16.
model = model.float().to(DEVICE)
print("Model dtype:", next(model.parameters()).dtype)
print("Model on   :", next(model.parameters()).device)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier

Model dtype: torch.float32
Model on   : cuda:0


In [18]:
_sample = {k: v.unsqueeze(0).to(DEVICE) for k, v in train_ds[0].items()}

with torch.no_grad():
    _out = model(
        input_ids=_sample["input_ids"],
        attention_mask=_sample["attention_mask"],
    )

_loss_fn    = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(DEVICE))
_check_loss = _loss_fn(_out.logits.float(), _sample["labels"].float())

print(f"Logits : {_out.logits.cpu()}")
print(f"Loss   : {_check_loss.item():.4f}")

assert not torch.isnan(_check_loss), "NaN loss before training starts — check data!"
assert _check_loss.item() < 20,      "Loss is unreasonably large — check pos_weight!"
print("\nSanity check passed ✓")

Logits : tensor([[-0.1177,  0.0409, -0.0043,  0.1977, -0.0116,  0.0188]])
Loss   : 0.7046

Sanity check passed ✓


In [22]:
total_steps  = math.ceil(len(train_ds) / BATCH_SIZE) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
print(f"Total steps: {total_steps}  |  Warmup steps: {warmup_steps}")

training_args = TrainingArguments(
    output_dir=str(MODEL_DIR),

    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    learning_rate=LR,
    weight_decay=0.01,
    warmup_steps=warmup_steps,

    fp16=False,
    bf16=USE_BF16,

    # ── gradient stability ────────────────────────────────────────────────────
    max_grad_norm=0.5,          # tighter than default 1.0
    
    # ── optimizer ─────────────────────────────────────────────────────────────
    # adamw_torch keeps optimizer states in fp32 regardless of model precision
    optim="adamw_torch",

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    label_names=["labels"],
    save_total_limit=1,
    report_to="none",
)

Total steps: 680  |  Warmup steps: 68


In [23]:
trainer = NaNGuardTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=build_compute_metrics(),
    pos_weight=pos_weight,
)

trainer.train()
print(f"\nNaN batches caught by guard: {trainer._nan_batches}")
tokenizer.save_pretrained(MODEL_DIR)
print("Tokenizer saved to", MODEL_DIR)

Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,F1 Narrative structure & quality,F1 Character & emotion,F1 Originality,F1 Immersion,F1 Thematic depth,F1 Writing style
1,1.271064,1.517671,0.178694,0.097246,0.000000,0.319328,0.000000,0.000000,0.000000,0.264151
2,1.207480,1.370363,0.305195,0.258104,0.464789,0.442907,0.126214,0.133333,0.098361,0.283019
3,1.046349,1.386358,0.331138,0.286465,0.551724,0.468254,0.128205,0.153846,0.085470,0.331288
4,0.951138,1.404251,0.336484,0.294297,0.551724,0.477733,0.129870,0.158273,0.093750,0.354430
5,0.886308,1.395810,0.329710,0.290301,0.547009,0.472868,0.124224,0.162162,0.088235,0.347305


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


NaN batches caught by guard: 0
Tokenizer saved to outputs/deberta_classifier


In [24]:
test_results = trainer.evaluate(test_ds)

print("\n── Test-set evaluation ──")
for k, v in test_results.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

Training Loss,Validation Loss,Epoch,F1 Micro,F1 Macro,F1 Narrative structure & quality,F1 Character & emotion,F1 Originality,F1 Immersion,F1 Thematic depth,F1 Writing style
0.886308,1.193475,5,0.308511,0.266398,0.482490,0.456929,0.086957,0.230769,0.078740,0.262500



── Test-set evaluation ──
  eval_loss: 1.1935
  eval_f1_micro: 0.3085
  eval_f1_macro: 0.2664
  eval_f1_Narrative Structure & Quality: 0.4825
  eval_f1_Character & Emotion: 0.4569
  eval_f1_Originality: 0.0870
  eval_f1_Immersion: 0.2308
  eval_f1_Thematic Depth: 0.0787
  eval_f1_Writing Style: 0.2625
